##  Step 1 — Verify GPU Runtime

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        " No GPU detected! Go to Runtime → Change runtime type → Hardware accelerator → GPU"
    )

gpu = torch.cuda.get_device_properties(0)
print(f" GPU: {gpu.name}")
print(f"   VRAM: {gpu.total_memory / 1e9:.1f} GB")
print(f"   CUDA: {torch.version.cuda}")

##  Step 2 — Install Dependencies

In [ ]:
!pip install -q \
    torch>=2.0.0 \
    transformers>=4.38.0 \
    datasets>=2.14.0 \
    peft>=0.8.0 \
    accelerate>=0.26.0 \
    bitsandbytes>=0.42.0 \
    huggingface_hub

print(" Dependencies installed")

##  Step 3 — Clone Your Repository

In [ ]:
import os

REPO_URL = "https://github.com/Jeel3011/llama-instruction-finetuning"  
REPO_DIR = "/content/llama-instruction-finetuning"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
    print(" Repository updated")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print(" Repository cloned")

os.chdir(REPO_DIR)
!ls -la

##  Step 4 — HuggingFace Login


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option A: Store token in Colab Secrets (recommended)
# Go to the key icon (🔑) in the left sidebar → Add secret → Name: HF_TOKEN
try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print(" Logged in via Colab Secrets")
except Exception:
    # Option B: Paste token directly (less secure)
    login()  # Will prompt you interactively
    print(" Logged in interactively")

##  Step 5 — Configure Training



In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from configs.config import get_config

# ─── Customize these ──────────────────────────────────────────────────────────
MAX_SAMPLES   = None     # None = full 52K Alpaca dataset | e.g. 500 for quick test
NUM_EPOCHS    = 3      # Number of training epochs
BATCH_SIZE    = 4        # Per-device batch size (reduce to 2 if OOM)
LORA_RANK     = 16       # LoRA rank (8 uses less memory, 16 is better quality)
MAX_LENGTH    = 512      # Max token length per sample
PUSH_TO_HUB   = False    # Set True to push adapter to HuggingFace Hub after training
HF_REPO_NAME  = "Jeel3011/llama-alpaca-finetuned"  # Your HF repo name
# ─────────────────────────────────────────────────────────────────────────────

config = get_config()
config.data.max_samples            = MAX_SAMPLES
config.training.num_epochs         = NUM_EPOCHS
config.training.batch_size         = BATCH_SIZE
config.lora.r                      = LORA_RANK
config.model.max_length            = MAX_LENGTH
config.hub.push_to_hub             = PUSH_TO_HUB
config.hub.repo_name               = HF_REPO_NAME
config.training.output_dir         = "/content/results"

print("Training Configuration:")
print(f"  Model:        {config.model.name}")
print(f"  Dataset:      {config.data.name}")
print(f"  Max samples:  {config.data.max_samples or 'Full dataset (~52K)'}")
print(f"  Epochs:       {config.training.num_epochs}")
print(f"  Batch size:   {config.training.batch_size}")
print(f"  LoRA rank:    {config.lora.r}")
print(f"  Push to Hub:  {config.hub.push_to_hub}")
print(f"  Output dir:   {config.training.output_dir}")

##  Step 6 — Prepare Dataset

In [ ]:
from src.data_prep import prepare_dataset

print("Loading and preparing dataset...")
dataset, tokenizer = prepare_dataset(config)

print(f"\n Dataset ready:")
print(f"   Train: {len(dataset['train']):,} samples")
print(f"   Test:  {len(dataset['test']):,} samples")
print(f"   Columns: {dataset['train'].column_names}")

# Preview a sample
print("\nSample prompt (first 300 chars):")
raw_sample = dataset['train'][0]
ids = raw_sample['input_ids'].tolist()
decoded = tokenizer.decode(ids[:100], skip_special_tokens=True)
print(decoded)

##  Step 7 — Load Model & Apply LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

print(f"Loading {config.model.name}...")
model = AutoModelForCausalLM.from_pretrained(
    config.model.name,
    dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=config.lora.r,
    lora_alpha=config.lora.lora_alpha,
    lora_dropout=config.lora.lora_dropout,
    target_modules=config.lora.target_modules,
    bias=config.lora.bias,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model ready with LoRA")

##  Step 8 — Train!

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=config.training.output_dir,
    num_train_epochs=config.training.num_epochs,
    per_device_train_batch_size=config.training.batch_size,
    gradient_accumulation_steps=config.training.gradient_accumulation_steps,
    learning_rate=config.training.learning_rate,
    warmup_steps=config.training.warmup_steps,
    logging_steps=config.training.logging_steps,
    save_steps=config.training.save_steps,
    save_total_limit=config.training.save_total_limit,
    eval_strategy="steps",
    eval_steps=config.training.eval_steps,
    fp16=True,
    push_to_hub=config.hub.push_to_hub,
    hub_model_id=config.hub.repo_name if config.hub.push_to_hub else None,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("🚀 Starting training... (this will take a while)")
trainer.train()
print("\n Training complete!")

##  Step 9 — Save the Adapter

In [ ]:
# Save locally in Colab
trainer.save_model(config.training.output_dir)
tokenizer.save_pretrained(config.training.output_dir)
print(f" Adapter saved to: {config.training.output_dir}")

# ── Option A: Save to Google Drive (recommended — persists after session) ──
import shutil

SAVE_TO_DRIVE = True   # Set False to skip

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dest = "/content/drive/MyDrive/llama-alpaca-finetuned"
    shutil.copytree(config.training.output_dir, drive_dest, dirs_exist_ok=True)
    print(f" Saved to Google Drive: {drive_dest}")

# ── Option B: Push to HuggingFace Hub ─────────────────────────────────────
if config.hub.push_to_hub:
    model.push_to_hub(config.hub.repo_name)
    tokenizer.push_to_hub(config.hub.repo_name)
    print(f" Pushed to HF Hub: https://huggingface.co/{config.hub.repo_name}")

##  Step 10 — Evaluate the Model

In [ ]:
import math

# Compute perplexity on test set
print("Computing evaluation metrics...")
eval_results = trainer.evaluate()

print(f"\n Evaluation Results:")
print(f"   Eval loss:   {eval_results['eval_loss']:.4f}")
print(f"   Perplexity:  {math.exp(eval_results['eval_loss']):.2f}")

##  Step 11 — Try the Fine-Tuned Model!

In [ ]:
from src.data_prep import format_instruction

def generate(instruction, input_text="", max_new_tokens=256, temperature=0.7):
    model.eval()
    example = {"instruction": instruction, "input": input_text, "output": ""}
    prompt = format_instruction(example)["text"]
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ── Test it with some instructions ────────────────────────────────────────────
test_prompts = [
    ("Give three tips for staying healthy.", ""),
    ("Write a short poem about the stars.", ""),
    ("What is the capital of France?", ""),
    ("Summarize the text below.", "The Amazon rainforest is the world's largest tropical rainforest, covering over 5.5 million square kilometers."),
]

for instruction, inp in test_prompts:
    response = generate(instruction, inp)
    print(f"\n Instruction: {instruction}")
    if inp:
        print(f"   Input:       {inp[:60]}...")
    print(f"Response:  {response}")
    print("-" * 60)